# UCS420 – Cognitive Computing | Assignment 10
## Heart Health Information Assistant

**Roll Number:** 1024170146  
**Domain selected:** Heart Health Information Assistant (even roll number)

### Objective
Develop a Python-based healthcare cognitive assistant that understands user queries, identifies their intent, retrieves relevant healthcare information, and generates appropriate responses.

**Safety note:** This is an academic information-retrieval demonstration, not a medical device. It cannot diagnose, prescribe, or rule out emergencies. If someone may be having a heart attack or stroke, contact emergency services immediately (112 in India).

## 1. Install and import libraries
Run this cell first. The remaining cells are designed to run from top to bottom in Google Colab.

In [ ]:
!pip -q install scikit-learn pandas matplotlib ipywidgets

In [ ]:
import re
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

print("All libraries imported successfully.")

## 2. Create the labelled query corpus
The corpus contains 50 sample queries across 6 intent categories.

In [ ]:
data = [
    # Symptoms (9)
    ("What are common symptoms of heart disease?", "symptoms"),
    ("Why do I feel chest discomfort sometimes?", "symptoms"),
    ("What does shortness of breath indicate?", "symptoms"),
    ("Can unusual tiredness be related to heart problems?", "symptoms"),
    ("What are signs of an irregular heartbeat?", "symptoms"),
    ("Why do my ankles swell?", "symptoms"),
    ("What are heart palpitations?", "symptoms"),
    ("Can dizziness be linked to heart conditions?", "symptoms"),
    ("What are warning signs of poor heart health?", "symptoms"),

    # Risk factors (8)
    ("Does smoking increase the risk of heart disease?", "risk_factors"),
    ("How does high cholesterol affect the heart?", "risk_factors"),
    ("Can diabetes increase cardiovascular risk?", "risk_factors"),
    ("Is family history a risk factor for heart disease?", "risk_factors"),
    ("Does obesity affect heart health?", "risk_factors"),
    ("How does stress influence cardiovascular health?", "risk_factors"),
    ("Is high blood pressure dangerous for the heart?", "risk_factors"),
    ("Does lack of exercise raise heart disease risk?", "risk_factors"),

    # Prevention (8)
    ("How can I keep my heart healthy?", "prevention"),
    ("What foods are good for heart health?", "prevention"),
    ("How much exercise supports a healthy heart?", "prevention"),
    ("What lifestyle changes reduce heart disease risk?", "prevention"),
    ("Is walking beneficial for the heart?", "prevention"),
    ("How can I reduce salt in my diet?", "prevention"),
    ("Why should I stop smoking for my heart?", "prevention"),
    ("How important is sleep for cardiovascular health?", "prevention"),

    # Blood pressure (8)
    ("What is blood pressure?", "blood_pressure"),
    ("What is considered high blood pressure?", "blood_pressure"),
    ("How can blood pressure be monitored at home?", "blood_pressure"),
    ("What does systolic pressure mean?", "blood_pressure"),
    ("What does diastolic pressure mean?", "blood_pressure"),
    ("Can stress temporarily raise blood pressure?", "blood_pressure"),
    ("Why is controlling blood pressure important?", "blood_pressure"),
    ("What should I do if my blood pressure reading is high?", "blood_pressure"),

    # Cholesterol (8)
    ("What is cholesterol?", "cholesterol"),
    ("What is LDL cholesterol?", "cholesterol"),
    ("What is HDL cholesterol?", "cholesterol"),
    ("Why is high LDL harmful?", "cholesterol"),
    ("How can cholesterol be checked?", "cholesterol"),
    ("What foods may help manage cholesterol?", "cholesterol"),
    ("What are triglycerides?", "cholesterol"),
    ("What is a lipid profile test?", "cholesterol"),

    # Healthy habits (9)
    ("How can I start exercising safely?", "healthy_habits"),
    ("What is a heart healthy diet?", "healthy_habits"),
    ("How does drinking alcohol affect the heart?", "healthy_habits"),
    ("Can meditation support heart health?", "healthy_habits"),
    ("How can I maintain a healthy weight?", "healthy_habits"),
    ("Why are regular health checkups useful?", "healthy_habits"),
    ("What are good habits for cardiovascular wellness?", "healthy_habits"),
    ("How can I manage stress better?", "healthy_habits"),
    ("How many hours of sleep should adults generally aim for?", "healthy_habits")
]

corpus = pd.DataFrame(data, columns=["Query", "Intent"])

print("Total queries:", len(corpus))
print("Number of intent categories:", corpus["Intent"].nunique())
display(corpus.groupby("Intent").size().rename("Number of queries").to_frame())
display(corpus.head())

## 3. Hierarchical taxonomy
There are 4 major categories, with at least 2 subcategories under each.

In [ ]:
taxonomy = {
    "Heart Health Basics": {
        "Symptoms": ["Common symptoms", "Warning signs"],
        "Risk factors": ["Modifiable risk factors", "Non-modifiable risk factors"]
    },
    "Clinical Measurements": {
        "Blood pressure": ["Understanding readings", "Monitoring and control"],
        "Cholesterol": ["LDL and HDL", "Lipid profile and triglycerides"]
    },
    "Prevention and Lifestyle": {
        "Prevention": ["Diet and nutrition", "Physical activity"],
        "Healthy habits": ["Sleep and stress", "Smoking and weight management"]
    },
    "Urgent Care and Safety": {
        "Emergency recognition": ["Possible heart attack", "Possible stroke"],
        "Getting help": ["Emergency services", "Professional medical advice"]
    }
}

for major_category, subcategories in taxonomy.items():
    print("\n" + major_category)
    for subcategory, topics in subcategories.items():
        print("  -", subcategory)
        for topic in topics:
            print("      •", topic)

## 4. Domain concepts and relationships
This section defines 10 domain concepts and 12 relationships.

In [ ]:
domain_concepts = {
    "Heart": "Organ responsible for pumping blood throughout the body.",
    "Blood pressure": "Force of blood against artery walls.",
    "Cholesterol": "A waxy substance carried in the blood.",
    "LDL": "Low-density lipoprotein, commonly called bad cholesterol.",
    "HDL": "High-density lipoprotein, commonly called good cholesterol.",
    "Smoking": "A behaviour that increases cardiovascular risk.",
    "Physical activity": "Body movement that supports cardiovascular fitness.",
    "Diet": "Food choices that influence cardiovascular health.",
    "Diabetes": "A condition associated with increased cardiovascular risk.",
    "Heart disease": "A group of conditions affecting the heart."
}

relationships = [
    ("High blood pressure", "increases risk of", "Heart disease"),
    ("Smoking", "increases risk of", "Heart disease"),
    ("Diabetes", "increases risk of", "Heart disease"),
    ("High LDL", "is associated with", "Atherosclerosis"),
    ("HDL", "helps transport", "Cholesterol"),
    ("Physical activity", "supports", "Heart health"),
    ("Balanced diet", "supports management of", "Cholesterol"),
    ("Balanced diet", "supports management of", "Blood pressure"),
    ("Heart disease", "may present with", "Chest discomfort"),
    ("Heart disease", "may present with", "Shortness of breath"),
    ("Blood pressure measurement", "helps monitor", "Blood pressure"),
    ("Lipid profile", "measures", "Cholesterol levels")
]

print("DOMAIN CONCEPTS")
for concept, definition in domain_concepts.items():
    print(f"- {concept}: {definition}")

print("\nDOMAIN RELATIONSHIPS")
for subject, relation, obj in relationships:
    print(f"- {subject} --{relation}--> {obj}")

print("\nTotal concepts:", len(domain_concepts))
print("Total relationships:", len(relationships))

## 5. NLP preprocessing
The preprocessing pipeline performs lowercase conversion, tokenization, stopword removal, and lemmatization.

In [ ]:
stop_words = set(ENGLISH_STOP_WORDS)

# A small rule-based lemmatizer keeps this notebook independent of downloaded language corpora.
irregular_lemmas = {
    "children": "child",
    "men": "man",
    "women": "woman",
    "teeth": "tooth",
    "feet": "foot",
    "diseases": "disease"
}

def lemmatize_word(word):
    if word in irregular_lemmas:
        return irregular_lemmas[word]

    if len(word) > 4 and word.endswith("ies"):
        return word[:-3] + "y"
    if len(word) > 4 and word.endswith("ves"):
        return word[:-3] + "f"
    if len(word) > 4 and word.endswith("ing"):
        base = word[:-3]
        if len(base) > 2 and base[-1] == base[-2]:
            base = base[:-1]
        return base
    if len(word) > 3 and word.endswith("ed"):
        base = word[:-2]
        if len(base) > 2 and base[-1] == base[-2]:
            base = base[:-1]
        return base
    if len(word) > 3 and word.endswith("s") and not word.endswith(("ss", "us", "is")):
        return word[:-1]

    return word

def preprocess_text(text):
    text = str(text).lower()                              # lowercase conversion
    tokens = re.findall(r"[a-z]+", text)                 # tokenization
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatize_word(token) for token in tokens] # lemmatization
    return " ".join(tokens)

corpus["Processed_Query"] = corpus["Query"].apply(preprocess_text)

display(corpus[["Query", "Processed_Query", "Intent"]].head(10))

## 6. TF-IDF feature extraction and Naive Bayes classifier
A stratified train/test split keeps examples from each intent in both sets.

In [ ]:
X = corpus["Processed_Query"]
y = corpus["Intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

tfidf = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

classifier = MultinomialNB()
classifier.fit(X_train_tfidf, y_train)

y_pred = classifier.predict(X_test_tfidf)

print("TF-IDF feature extraction completed.")
print("Naive Bayes classifier trained successfully.")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 7. Model evaluation
The model is evaluated using accuracy, weighted precision, weighted recall, weighted F1-score, and a confusion matrix.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("MODEL EVALUATION")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

labels = classifier.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(9, 7))
display_matrix = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
display_matrix.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
plt.title("Confusion Matrix – Heart Health Intent Classifier")
plt.tight_layout()
plt.show()

## 8. Heart-health knowledge base
The knowledge base contains general educational information for each supported intent.

In [ ]:
knowledge_base = {
    "symptoms": {
        "title": "Heart-health symptoms",
        "response": (
            "Possible symptoms associated with heart problems include chest discomfort, "
            "shortness of breath, unusual fatigue, dizziness, palpitations, and swelling "
            "in the legs or ankles. These symptoms can have many causes. New, persistent, "
            "or concerning symptoms should be assessed by a healthcare professional."
        )
    },
    "risk_factors": {
        "title": "Cardiovascular risk factors",
        "response": (
            "Risk factors can include high blood pressure, high LDL cholesterol, diabetes, "
            "smoking, excess body weight, physical inactivity, and family history. Some "
            "factors can be modified through lifestyle changes and appropriate medical care."
        )
    },
    "prevention": {
        "title": "Heart disease prevention",
        "response": (
            "Helpful steps include eating a balanced diet with vegetables, fruits and whole "
            "grains; limiting excess salt and highly processed foods; being physically active; "
            "avoiding tobacco; managing stress; sleeping adequately; and attending recommended "
            "health checkups."
        )
    },
    "blood_pressure": {
        "title": "Blood pressure",
        "response": (
            "Blood pressure measures the force of blood against artery walls. Systolic pressure "
            "is the pressure when the heart beats, while diastolic pressure is the pressure "
            "between beats. Interpret readings with a healthcare professional because repeated "
            "measurements and personal context matter."
        )
    },
    "cholesterol": {
        "title": "Cholesterol",
        "response": (
            "Cholesterol is a substance carried in the blood. LDL is often called bad cholesterol "
            "because higher levels can contribute to plaque buildup in arteries. HDL is often "
            "called good cholesterol because it helps carry cholesterol away. A lipid profile "
            "can measure cholesterol and triglyceride levels."
        )
    },
    "healthy_habits": {
        "title": "Heart-healthy habits",
        "response": (
            "Build habits gradually: choose regular enjoyable physical activity, balanced meals, "
            "adequate sleep, tobacco avoidance, stress-management practices, and routine preventive "
            "checkups. Ask a healthcare professional what activity level is appropriate if you "
            "have a health condition or symptoms."
        )
    }
}

print("Knowledge base created for", len(knowledge_base), "intent categories.")
for intent, item in knowledge_base.items():
    print("-", intent, ":", item["title"])

## 9. Reasoning rules and integrated assistant
The assistant applies emergency escalation first, then unsupported-query handling, intent classification, low-confidence handling, and knowledge retrieval.

In [ ]:
emergency_phrases = [
    "chest pain", "crushing chest", "pressure in chest",
    "shortness of breath", "difficulty breathing", "cannot breathe",
    "cold sweat", "fainting", "face drooping", "arm weakness",
    "speech difficulty", "sudden weakness"
]

supported_keywords = {
    "heart", "cardiac", "blood", "pressure", "cholesterol", "ldl", "hdl",
    "pulse", "heartbeat", "smoking", "diabetes", "exercise", "diet",
    "symptom", "chest", "breath", "artery", "cardiovascular",
    "triglyceride", "lipid", "stress", "sleep", "weight", "systolic",
    "diastolic", "palpitation", "dizziness", "ankle", "swelling",
    "meditation", "fatigue", "obesity"
}

def is_emergency(query):
    query = query.lower()
    return any(phrase in query for phrase in emergency_phrases)

def is_supported(query):
    words = set(re.findall(r"[a-z]+", query.lower()))
    return bool(words.intersection(supported_keywords))

def detect_intent_hint(query):
    """Use clear domain phrases to resolve obvious intents before ML prediction."""
    q = query.lower()

    if any(term in q for term in [
        "blood pressure", "systolic", "diastolic", " bp "
    ]):
        return "blood_pressure"

    if any(term in q for term in [
        "cholesterol", "ldl", "hdl", "triglyceride", "lipid profile"
    ]):
        if "risk" in q or "increase" in q:
            return "risk_factors"
        return "cholesterol"

    if any(term in q for term in [
        "symptom", "chest discomfort", "palpitation", "dizziness",
        "shortness of breath", "ankle swell", "unusual fatigue"
    ]):
        return "symptoms"

    if any(term in q for term in [
        "risk factor", "increase the risk", "increases risk",
        "family history", "smoking increase", "diabetes increase",
        "obesity affect"
    ]):
        return "risk_factors"

    if any(term in q for term in [
        "meditation", "healthy habit", "health checkup", "sleep",
        "manage stress", "stress better", "healthy weight",
        "exercise safely", "drinking alcohol"
    ]):
        return "healthy_habits"

    if any(term in q for term in [
        "keep my heart healthy", "prevent", "prevention",
        "reduce heart disease risk", "foods are good for heart",
        "reduce salt", "walking beneficial", "lifestyle changes"
    ]):
        return "prevention"

    return None

def healthcare_assistant(query):
    query = str(query).strip()

    # Rule 1: Empty-query handling.
    if not query:
        return "Please enter a question about heart health."

    # Rule 2: Emergency escalation takes priority over classification.
    if is_emergency(query):
        return (
            "⚠️ POSSIBLE MEDICAL EMERGENCY\\n\\n"
            "The query mentions symptoms that may require urgent medical attention. "
            "Call emergency services immediately (112 in India) or go to the nearest "
            "emergency department. Do not wait for this assistant to assess the symptoms. "
            "This is not a diagnosis."
        )

    # Rule 3: Unsupported-query handling.
    if not is_supported(query):
        return (
            "I am designed to answer general heart-health questions. You can ask about "
            "symptoms, cardiovascular risk factors, prevention, blood pressure, cholesterol, "
            "or healthy habits."
        )

    processed_query = preprocess_text(query)
    if not processed_query:
        return "I could not understand the question. Please rephrase it."

    query_vector = tfidf.transform([processed_query])
    probabilities = classifier.predict_proba(query_vector)[0]
    best_index = probabilities.argmax()
    model_intent = classifier.classes_[best_index]
    confidence = float(probabilities[best_index])

    # Rule 4: Clear keyword-based reasoning can resolve common, unambiguous intents.
    intent_hint = detect_intent_hint(query)
    if intent_hint is not None:
        predicted_intent = intent_hint
        decision_method = "domain reasoning rule"
    else:
        predicted_intent = model_intent
        decision_method = "Naive Bayes classifier"

    # Rule 5: Low-confidence handling when no clear reasoning rule applies.
    if confidence < 0.30 and intent_hint is None:
        return (
            "I am not sufficiently confident about the intent of this question. "
            "Please rephrase it or ask about symptoms, risk factors, prevention, "
            "blood pressure, cholesterol, or healthy habits."
        )

    # Rule 6: Retrieve information from the knowledge base.
    response = knowledge_base[predicted_intent]["response"]
    return (
        f"**Detected intent:** {predicted_intent.replace('_', ' ').title()}\\n\\n"
        f"**Decision method:** {decision_method}\\n\\n"
        f"**Model confidence:** {confidence:.2f}\\n\\n"
        f"{response}\\n\\n"
        "_For personal symptoms, test results, or treatment decisions, consult a qualified "
        "healthcare professional._"
    )

print("Healthcare Cognitive Assistant is ready.")

## 10. Test the assistant
The tests include every intent category, a differently worded query, an unsupported query, a vague/low-confidence-style query, and an emergency query.

In [ ]:
test_queries = [
    ("Intent: Symptoms", "What are common symptoms of heart disease?"),
    ("Intent: Risk factors", "Does smoking increase the risk of heart disease?"),
    ("Intent: Prevention", "How can I keep my heart healthy?"),
    ("Intent: Blood pressure", "What is systolic pressure?"),
    ("Intent: Cholesterol", "What is LDL cholesterol?"),
    ("Intent: Healthy habits", "Can meditation support heart health?"),
    ("Differently worded query", "My blood pressure number seems elevated. What does it mean?"),
    ("Unsupported query", "How do I repair a bicycle chain?"),
    ("Vague query", "Tell me about it."),
    ("Emergency query", "I have crushing chest pain and cold sweat.")
]

for test_name, query in test_queries:
    print("\n" + "=" * 85)
    print(test_name)
    print("User:", query)
    print("\nAssistant:")
    print(healthcare_assistant(query))
print("\nAll test queries completed.")

## 11. Interactive assistant
Run this cell to open the interactive interface. You can ask multiple questions one after another.

In [ ]:
question_box = widgets.Text(
    value="",
    placeholder="Ask a heart-health question...",
    description="Question:",
    layout=widgets.Layout(width="85%")
)

ask_button = widgets.Button(
    description="Ask Assistant",
    button_style="primary",
    icon="comment"
)

clear_button = widgets.Button(
    description="Clear",
    button_style="warning"
)

output_area = widgets.Output()

def on_ask_clicked(_):
    query = question_box.value
    with output_area:
        display(Markdown("**You:** " + (query if query.strip() else "(empty question)")))
        display(Markdown("**Assistant:**\n\n" + healthcare_assistant(query)))
        display(Markdown("---"))
    question_box.value = ""

def on_clear_clicked(_):
    output_area.clear_output()

ask_button.on_click(on_ask_clicked)
clear_button.on_click(on_clear_clicked)

display(widgets.HTML("<h3>❤️ Heart Health Information Assistant</h3>"))
display(widgets.HTML(
    "<p>Ask general questions about heart health. This tool is for education only. "
    "For a possible emergency in India, call 112.</p>"
))
display(widgets.VBox([
    question_box,
    widgets.HBox([ask_button, clear_button]),
    output_area
]))

## 12. Conclusion

The Heart Health Information Assistant was developed using Python, Natural Language Processing, and machine learning.

The notebook includes:
1. A labelled corpus of 50 queries across 6 intent categories.
2. A hierarchical taxonomy with 4 major categories and 2 subcategories under each.
3. 10 domain concepts and 12 relationships.
4. Lowercase conversion, tokenization, stopword removal, and lemmatization.
5. TF-IDF feature extraction and a Multinomial Naive Bayes classifier.
6. Accuracy, precision, recall, F1-score, and confusion-matrix evaluation.
7. A domain-specific heart-health knowledge base.
8. Rule-based handling for emergencies, low confidence, unsupported queries, and empty input.
9. Integration of NLP, intent classification, reasoning, and knowledge retrieval.
10. An interactive multi-query interface in Google Colab.

**Limitations:** The dataset is small and illustrative, so the evaluation scores are only a classroom demonstration. Keyword-based emergency detection cannot identify every emergency. The assistant is not clinically validated and must not be used for diagnosis or treatment decisions.